# Export annual temperature anomaly rasters

This notebook processes ERA5 2 m temperature data for the Mont Blanc region. It converts temperature from Kelvin to Celsius, calculates annual mean temperatures, compares each year against a 1981–2010 baseline, and exports annual temperature anomaly rasters for mapping and visualisation.

In [ ]:
# Install required packages if running in a fresh environment
# Uncomment if needed:
# !pip install xarray rioxarray numpy netcdf4 h5py

In [ ]:
import xarray as xr
import rioxarray

In [ ]:
# Load ERA5 2 m temperature data
temp = xr.open_dataset("2m_temp.nc")

print(temp)

In [ ]:
# Convert 2 m temperature from Kelvin to Celsius
temp_c = temp["t2m"] - 273.15
temp_c.attrs["units"] = "°C"

In [ ]:
# Calculate annual mean temperature
annual_temp = temp_c.resample(valid_time="YE").mean()

In [ ]:
# Calculate the 1981–2010 baseline mean
baseline = annual_temp.sel(valid_time=slice("1981", "2010")).mean("valid_time")

# Calculate annual anomalies relative to the baseline
anomaly = annual_temp - baseline
anomaly.attrs["units"] = "°C anomaly relative to 1981–2010"

In [ ]:
# Assign coordinate reference system for raster export
anomaly = anomaly.rio.write_crs("EPSG:4326")

In [ ]:
# Export one GeoTIFF per year
years = range(1950, 2026)

for year in years:
    annual_anomaly = anomaly.sel(valid_time=str(year))
    output_path = f"temp_anomaly_{year}.tif"
    annual_anomaly.rio.to_raster(output_path)
    print(f"Saved {output_path}")